In [7]:
import pandas as pd
import numpy as np
from pathlib import Path

In [8]:
DATA_DIR = Path("../data/raw")   
OUT_DIR  = Path("../data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ✅ FIX 1 : fonction load définie
def load(name):
    return pd.read_csv(DATA_DIR / f"cleaned_{name}.csv")

In [9]:
# ── CHARGEMENT ───────────────────────────────────────────────────────────────
lap_times    = load("lap_times")
pit_stops    = load("pit_stops")
qualifying   = load("qualifying")
results      = load("results")
races        = load("races")
driver_std   = load("driver_standings")
constr_std   = load("constructor_standings")
status       = load("status")

print(f"✅ lap_times  : {len(lap_times):,} lignes")
print(f"✅ pit_stops  : {len(pit_stops):,} lignes")
print(f"✅ results    : {len(results):,} lignes")
print(f"✅ races      : {len(races):,} lignes")

✅ lap_times  : 551,742 lignes
✅ pit_stops  : 10,089 lignes
✅ results    : 26,080 lignes
✅ races      : 1,101 lignes


In [13]:
# DIAGNOSTIC : voir les colonnes disponibles
print("Colonnes lap_times:", lap_times.columns.tolist())
print("\nAperçu lap_times:")
lap_times.head(3)

Colonnes lap_times: ['raceId', 'driverId', 'lap', 'position', 'lap_time']

Aperçu lap_times:


,raceId,driverId,lap,position,lap_time
0,841,20,1,1,0 days 00:01:38.109000
1,841,20,2,1,0 days 00:01:33.006000
2,841,20,3,1,0 days 00:01:32.713000


In [15]:
# ── ÉTAPE 1 : BASE lapTimes ─────────────────────────────────────────────────
df = lap_times[["raceId","driverId","lap","lap_time","position"]].copy()

# Convertir string → timedelta → millisecondes
df["lap_time"] = pd.to_timedelta(df["lap_time"])  # ← AJOUTÉ
df["last_lap_time_ms"] = df["lap_time"].dt.total_seconds() * 1000
df = df.drop(columns=["lap_time"])

df = df.sort_values(["raceId","driverId","lap"]).reset_index(drop=True)

# Rolling stats sur les tours précédents
grp = df.groupby(["raceId","driverId"])
df["avg_lap_time_ms"]  = grp["last_lap_time_ms"].transform(lambda x: x.expanding().mean())
df["best_lap_time_ms"] = grp["last_lap_time_ms"].transform(lambda x: x.expanding().min())
df["lap_time_std"]     = grp["last_lap_time_ms"].transform(lambda x: x.expanding().std().fillna(0))

# total_laps par course
race_laps = results.groupby("raceId")["laps"].max().reset_index()
race_laps.columns = ["raceId","total_laps"]
df = df.merge(race_laps, on="raceId", how="left")
df["laps_remaining"]    = df["total_laps"] - df["lap"]
df["race_progress_pct"] = (df["lap"] / df["total_laps"] * 100).round(2)

# current_position (déjà dans lap_times)
df = df.rename(columns={"position": "current_position"})

# is_fastest_lap : pilote avec le meilleur temps à ce tour dans la course
best_per_lap = (df.groupby(["raceId","lap"])["last_lap_time_ms"]
                  .min().reset_index()
                  .rename(columns={"last_lap_time_ms":"race_best_ms"}))
df = df.merge(best_per_lap, on=["raceId","lap"], how="left")
df["is_fastest_lap"] = (df["last_lap_time_ms"] == df["race_best_ms"]).astype(int)
df.drop(columns=["race_best_ms"], inplace=True)

print(f"✅ Étape 1 OK — {len(df):,} lignes, {df.shape[1]} colonnes")

✅ Étape 1 OK — 551,742 lignes, 12 colonnes


In [16]:
# ── ÉTAPE 2 : PIT STOPS (vectorisé) ─────────────────────────────────────────
pit = pit_stops[["raceId","driverId","lap","stop"]].rename(
      columns={"lap":"pit_lap","stop":"pit_stop_n"})

# Croiser chaque tour avec tous les pit stops du même pilote/course
df_pit = df[["raceId","driverId","lap"]].merge(pit, on=["raceId","driverId"], how="left")
df_pit["pit_done"] = (df_pit["pit_lap"] <= df_pit["lap"]).astype(int)

pit_counts = (df_pit[df_pit["pit_done"]==1]
              .groupby(["raceId","driverId","lap"])
              .agg(nb_pit_stops=("pit_stop_n","max"),
                   last_pit_lap=("pit_lap","max"))
              .reset_index())

df = df.merge(pit_counts, on=["raceId","driverId","lap"], how="left")
df["nb_pit_stops"]   = df["nb_pit_stops"].fillna(0).astype(int)
df["last_pit_lap"]   = df["last_pit_lap"].fillna(0).astype(int)
df["laps_since_pit"] = df["lap"] - df["last_pit_lap"]

print(f"✅ Étape 2 OK — {len(df):,} lignes, {df.shape[1]} colonnes")

✅ Étape 2 OK — 551,742 lignes, 15 colonnes


In [17]:
# ── ÉTAPE 3 : QUALIFYING ─────────────────────────────────────────────────────
qual = qualifying[["raceId","driverId","position"]].rename(
       columns={"position":"grid_position"})
df = df.merge(qual, on=["raceId","driverId"], how="left")
df["grid_position"] = df["grid_position"].fillna(20).astype(int)

if "current_position" in df.columns:
    df["positions_gained"] = df["grid_position"] - df["current_position"]

print(f"✅ Étape 3 OK — {len(df):,} lignes, {df.shape[1]} colonnes")

✅ Étape 3 OK — 551,742 lignes, 17 colonnes


In [18]:
# ── ÉTAPE 4 : DRIVER STANDINGS (standings de la course précédente) ───────────
races_info = races[["raceId","year","round"]].sort_values(["year","round"])
df = df.merge(races_info, on="raceId", how="left")

ds = driver_std[["raceId","driverId","points","position","wins"]].copy()
ds.columns = ["raceId","driverId","driver_season_points","driver_season_pos","driver_wins_season"]
ds = ds.merge(races_info, on="raceId").rename(columns={"year":"ds_year","round":"ds_round"})
ds["next_round"] = ds["ds_round"] + 1

df = df.merge(
    ds[["ds_year","next_round","driverId","driver_season_points","driver_season_pos","driver_wins_season"]],
    left_on=["year","round","driverId"],
    right_on=["ds_year","next_round","driverId"],
    how="left"
)
df.drop(columns=["ds_year","next_round"], errors="ignore", inplace=True)
df[["driver_season_points","driver_season_pos","driver_wins_season"]] = \
    df[["driver_season_points","driver_season_pos","driver_wins_season"]].fillna(0)

print(f"✅ Étape 4 OK — {len(df):,} lignes, {df.shape[1]} colonnes")

✅ Étape 4 OK — 551,742 lignes, 22 colonnes


In [19]:
# ── ÉTAPE 5 : CONSTRUCTOR STANDINGS ─────────────────────────────────────────
res_c = results[["raceId","driverId","constructorId"]].drop_duplicates()
df = df.merge(res_c, on=["raceId","driverId"], how="left")

cs = constr_std[["raceId","constructorId","points","position"]].copy()
cs.columns = ["raceId","constructorId","constructor_points","constructor_pos"]
cs = cs.merge(races_info, on="raceId").rename(columns={"year":"cs_year","round":"cs_round"})
cs["next_round"] = cs["cs_round"] + 1

df = df.merge(
    cs[["cs_year","next_round","constructorId","constructor_points","constructor_pos"]],
    left_on=["year","round","constructorId"],
    right_on=["cs_year","next_round","constructorId"],
    how="left"
)
df.drop(columns=["cs_year","next_round"], errors="ignore", inplace=True)
df[["constructor_points","constructor_pos"]] = \
    df[["constructor_points","constructor_pos"]].fillna(0)

print(f"✅ Étape 5 OK — {len(df):,} lignes, {df.shape[1]} colonnes")

✅ Étape 5 OK — 551,742 lignes, 25 colonnes


In [22]:
# DIAGNOSTIC : colonnes results
print("Colonnes results:", results.columns.tolist())
print("\nAperçu results:")
results.head(3)

Colonnes results: ['resultId', 'raceId', 'driverId', 'constructorId', 'grid_position', 'position', 'points', 'laps', 'fastestLap', 'fastestLapRank', 'fastestLapTime', 'fastestLapSpeed', 'statusId']

Aperçu results:


,resultId,raceId,driverId,constructorId,grid_position,position,points,laps,fastestLap,fastestLapRank,fastestLapTime,fastestLapSpeed,statusId
0,1,18,1,1,1,1,10.0,58,39,2,0 days 00:01:27.452000,218.300,1
1,2,18,2,2,5,2,8.0,58,41,3,0 days 00:01:27.739000,217.586,1
2,3,18,3,3,7,3,6.0,58,41,5,0 days 00:01:28.090000,216.719,1


In [23]:
# ── ÉTAPE 6 : HISTORIQUE CIRCUIT ─────────────────────────────────────────────

# Éviter doublon circuitId
if "circuitId" not in df.columns:
    df = df.merge(races[["raceId","circuitId"]], on="raceId", how="left")
elif "circuitId_x" in df.columns:
    df = df.rename(columns={"circuitId_x": "circuitId"})
    df.drop(columns=["circuitId_y"], errors="ignore", inplace=True)

# Historique pilote sur ce circuit
res_hist = (results[["raceId","driverId","constructorId","position"]]
            .merge(races[["raceId","circuitId"]], on="raceId"))

circuit_stats = (res_hist.groupby(["driverId","circuitId"])
                         .agg(driver_avg_pos_circuit=("position","mean"),
                              driver_wins_circuit=("position", lambda x: (x==1).sum()))
                         .reset_index())
df = df.merge(circuit_stats, on=["driverId","circuitId"], how="left")
df["driver_avg_pos_circuit"] = df["driver_avg_pos_circuit"].fillna(10.0)
df["driver_wins_circuit"]    = df["driver_wins_circuit"].fillna(0).astype(int)

# Historique constructeur sur ce circuit
constr_circ = (res_hist.groupby(["constructorId","circuitId"])
                       .agg(constructor_avg_circuit=("position","mean"))
                       .reset_index())
df = df.merge(constr_circ, on=["constructorId","circuitId"], how="left")
df["constructor_avg_circuit"] = df["constructor_avg_circuit"].fillna(10.0)

print(f"✅ Étape 6 OK — {len(df):,} lignes, {df.shape[1]} colonnes")

✅ Étape 6 OK — 551,742 lignes, 29 colonnes


In [29]:
# ── ÉTAPE 7 : TARGET (avec protection contre double exécution)
if "final_position" in df.columns:
    print("⚠️  Target déjà ajouté, étape ignorée")
else:
    dnf_ids = status[status["status"].str.contains(
        "Accident|Collision|Engine|Mechanical|Retired|DNF|Disqualified",
        case=False, na=False)]["statusId"].tolist()

    target = (results[["raceId","driverId","position","statusId"]]
              .rename(columns={"position":"final_position"}))
    target["is_dnf"] = target["statusId"].isin(dnf_ids).astype(int)

    df = df.merge(target[["raceId","driverId","final_position","is_dnf"]],
                  on=["raceId","driverId"], how="left")

    print(f"✅ Étape 7 OK — {len(df):,} lignes, {df.shape[1]} colonnes")

✅ Étape 7 OK — 551,742 lignes, 35 colonnes


In [28]:
# DIAGNOSTIC après étape 7
print("Colonnes dans df:", df.columns.tolist())
print("\nEst-ce que 'final_position' existe ?", "final_position" in df.columns)
print("\nAperçu df:")
df.head(3)

Colonnes dans df: ['raceId', 'driverId', 'lap', 'current_position', 'last_lap_time_ms', 'avg_lap_time_ms', 'best_lap_time_ms', 'lap_time_std', 'total_laps', 'laps_remaining', 'race_progress_pct', 'is_fastest_lap', 'nb_pit_stops', 'last_pit_lap', 'laps_since_pit', 'grid_position', 'positions_gained', 'year', 'round', 'driver_season_points', 'driver_season_pos', 'driver_wins_season', 'constructorId', 'constructor_points', 'constructor_pos', 'circuitId', 'driver_avg_pos_circuit', 'driver_wins_circuit', 'constructor_avg_circuit', 'final_position_x', 'is_dnf_x', 'final_position_y', 'is_dnf_y']

Est-ce que 'final_position' existe ? False

Aperçu df:


,raceId,driverId,lap,current_position,last_lap_time_ms,avg_lap_time_ms,best_lap_time_ms,lap_time_std,total_laps,laps_remaining,...,constructor_points,constructor_pos,circuitId,driver_avg_pos_circuit,driver_wins_circuit,constructor_avg_circuit,final_position_x,is_dnf_x,final_position_y,is_dnf_y
0,1,1,1,13,109088.0,109088.000000,109088.0,0.000000,58,57,...,0.0,0.0,1,4.933333,2,8.576923,20,1,20,1
1,1,1,2,12,93740.0,101414.000000,93740.0,10852.674878,58,56,...,0.0,0.0,1,4.933333,2,8.576923,20,1,20,1
2,1,1,3,11,91600.0,98142.666667,91600.0,9539.137347,58,55,...,0.0,0.0,1,4.933333,2,8.576923,20,1,20,1


In [30]:
# ── EXPORT ────────────────────────────────────────────────────────────────────
df = df.dropna(subset=["final_position"])

# Colonnes finales dans l'ordre logique
FINAL_COLS = [
    # Identifiants
    "raceId", "driverId", "constructorId", "circuitId", "year", "round",
    # Contexte course
    "lap", "total_laps", "laps_remaining", "race_progress_pct",
    # Position
    "current_position", "grid_position", "positions_gained",
    # Performance
    "last_lap_time_ms", "avg_lap_time_ms", "best_lap_time_ms", "lap_time_std",
    "is_fastest_lap",
    # Pit stops
    "nb_pit_stops", "last_pit_lap", "laps_since_pit",
    # Contexte saison
    "driver_season_points", "driver_season_pos", "driver_wins_season",
    "constructor_points", "constructor_pos",
    # Historique circuit
    "driver_avg_pos_circuit", "driver_wins_circuit", "constructor_avg_circuit",
    # Target
    "final_position", "is_dnf",
]

# Garder seulement les colonnes qui existent
FINAL_COLS = [c for c in FINAL_COLS if c in df.columns]
df_final = df[FINAL_COLS].copy()

# Vérification valeurs manquantes
missing = df_final.isnull().sum()
missing = missing[missing > 0]
if len(missing) > 0:
    print("⚠️  Valeurs manquantes :")
    print(missing)
else:
    print("✅ Aucune valeur manquante")

# Export
out_path = OUT_DIR / "training_dataset.csv"
df_final.to_csv(out_path, index=False)

print(f"\nSauvegardé : {out_path}")
print(f"   {len(df_final):,} lignes × {len(df_final.columns)} colonnes")
print(f"\nAperçu :")
df_final.head()

✅ Aucune valeur manquante

Sauvegardé : ../data/processed/training_dataset.csv
   551,742 lignes × 31 colonnes

Aperçu :


,raceId,driverId,constructorId,circuitId,year,round,lap,total_laps,laps_remaining,race_progress_pct,...,driver_season_points,driver_season_pos,driver_wins_season,constructor_points,constructor_pos,driver_avg_pos_circuit,driver_wins_circuit,constructor_avg_circuit,final_position,is_dnf
0,1,1,1,1,2009,1,1,58,57,1.72,...,0.0,0.0,0.0,0.0,0.0,4.933333,2,8.576923,20,1
1,1,1,1,1,2009,1,2,58,56,3.45,...,0.0,0.0,0.0,0.0,0.0,4.933333,2,8.576923,20,1
2,1,1,1,1,2009,1,3,58,55,5.17,...,0.0,0.0,0.0,0.0,0.0,4.933333,2,8.576923,20,1
3,1,1,1,1,2009,1,4,58,54,6.90,...,0.0,0.0,0.0,0.0,0.0,4.933333,2,8.576923,20,1
4,1,1,1,1,2009,1,5,58,53,8.62,...,0.0,0.0,0.0,0.0,0.0,4.933333,2,8.576923,20,1
